In [1]:
import sys, os
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf')
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf/tdmpc/src')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['MUJOCO_GL'] = 'egl'

In [2]:
import time
import torch
import matplotlib.pyplot as plt
from env import make_env
from algorithm.helper import Episode, ReplayBuffer
from scripts.train_o2 import make_cfg
from o2.training_utils import set_seed
from o2_base.agent import TDMPC_Dec

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
cfg = make_cfg(
    'cheetah-run', seed=1,
    exp_name='o2_base',
    action_repeat=1,
    std_schedule='linear(0.5,0.05,10000)',
    horizon_schedule='linear(1,5,10000)',
    train_steps=20000,
    seed_steps=3000,
    horizon=5,
    dec_hidden_dim=256,
    dec_num_layers=2,
    latent_num_samples=32,
    latent_num_elites=8,
    dcem_iterations=5,
    lml_temperature=1,
    dcem_batch_size=64,
    decoder_updates=50,
)

PHASE2_START = 10000          # step at which acting switches to decoder + decoder training starts
LATENT_ACTION_DIM = 100

In [4]:
set_seed(cfg.seed)
env = make_env(cfg)
cfg.latent_action_dim = LATENT_ACTION_DIM
agent  = TDMPC_Dec(cfg)
buffer = ReplayBuffer(cfg)
print(agent.decoder)

MLPDecoder(
  (net): Sequential(
    (0): Linear(in_features=150, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=30, bias=True)
  )
)


In [ ]:
@torch.no_grad()
def decoder_act(agent, obs, step, eval_mode=False):
    """Act by running DCEM (no grad), sampling u from the final distribution (mean if eval_mode), decoding; first action only."""
    obs = torch.tensor(obs, dtype=torch.float32, device=agent.device).unsqueeze(0)
    u_mean, u_std, _ = agent.dcem(obs, step)
    u = u_mean if eval_mode else u_mean + u_std * torch.randn_like(u_mean)
    return agent.decoder(u, agent.model.h(obs))[0, 0]


In [6]:
rewards, phases = [], []
start_time = time.time()

for step in range(0, cfg.train_steps + cfg.episode_length, cfg.episode_length):
    phase2 = step >= PHASE2_START

    obs = env.reset()
    episode = Episode(cfg, obs)
    while not episode.done:
        if phase2:
            action = decoder_act(agent, obs, step)
        else:
            action = agent.plan(obs, step=step, t0=episode.first)   
        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    buffer += episode

    told_metrics = {}
    if step >= cfg.seed_steps:
        num_updates = cfg.seed_steps if step == cfg.seed_steps else cfg.episode_length
        for i in range(num_updates):
            told_metrics = agent.update(buffer, step + i)

    dec_metrics = {}
    if phase2:
        for _ in range(cfg.decoder_updates):
            obs_batch, *_ = buffer.sample()
            dec_metrics = agent.update_decoder(obs_batch[:cfg.dcem_batch_size], step)

    rewards.append(episode.cumulative_reward)
    phases.append(2 if phase2 else 1)
    print(f'step {step:>6}  phase {phases[-1]}  reward {rewards[-1]:7.1f}  '
          f'value_loss {told_metrics.get("value_loss", float("nan")):8.3f}  '
          f'dec_loss {dec_metrics.get("decoder_loss", float("nan")):8.3f}  '
          f'time {time.time() - start_time:6.0f}s')

step      0  phase 1  reward     3.9  value_loss      nan  dec_loss      nan  time      1s
step   1000  phase 1  reward     4.6  value_loss      nan  dec_loss      nan  time      1s
step   2000  phase 1  reward     3.4  value_loss      nan  dec_loss      nan  time      2s
step   3000  phase 1  reward     4.6  value_loss    0.000  dec_loss      nan  time    188s
step   4000  phase 1  reward    12.6  value_loss    0.001  dec_loss      nan  time    266s
step   5000  phase 1  reward     7.5  value_loss    0.001  dec_loss      nan  time    346s
step   6000  phase 1  reward    31.4  value_loss    0.002  dec_loss      nan  time    424s
step   7000  phase 1  reward    28.3  value_loss    0.004  dec_loss      nan  time    503s
step   8000  phase 1  reward    50.4  value_loss    0.007  dec_loss      nan  time    584s
step   9000  phase 1  reward    33.7  value_loss    0.007  dec_loss      nan  time    666s
step  10000  phase 2  reward     1.6  value_loss    0.005  dec_loss   -3.550  time    761s

In [ ]:
steps = [i * cfg.episode_length for i in range(len(rewards))]
plt.plot(steps, rewards)
plt.axvline(PHASE2_START, color='k', linestyle='--')
plt.xlabel('step')
plt.ylabel('episode reward')
plt.show()